/Volumes/workspace/default/capstoneproject/SP500_ETF_Data_Raw.csv

In [0]:
%sql
create or replace temp view spx_raw
using csv
options (
  path = "/Volumes/workspace/default/capstoneproject/SP500_ETF_Data_V2_Raw.csv",
  header = true,
  inferSchema = "true"
  );

In [0]:
%sql
select * from spx_raw;

In [0]:
from pyspark.sql import functions as F

path = "/Volumes/workspace/default/capstoneproject/SP500_ETF_Data_V2_Raw.csv"

df_raw = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(path))

display(df_raw)
print(df_raw.columns)


In [0]:
rename_map = {
  "('Date',_'')": "Date",
  "('Open',_'^GSPC')": "Open",
  "('High',_'^GSPC')": "High",
  "('Low',_'^GSPC')": "Low",
  "('Close',_'^GSPC')": "Close",
  "('Volume',_'^GSPC')": "Volume",
}

df = df_raw
for old, new in rename_map.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)

df = (df
      .withColumn("Date", F.to_date("Date"))
      .withColumn("Open", F.col("Open").cast("double"))
      .withColumn("High", F.col("High").cast("double"))
      .withColumn("Low", F.col("Low").cast("double"))
      .withColumn("Close", F.col("Close").cast("double"))
      .withColumn("Volume", F.col("Volume").cast("double"))
      .orderBy("Date")
)

display(df)
print(df.columns)


In [0]:
%sql
DESCRIBE spx_raw;

In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

schema_str = StructType([
    StructField("Date", StringType()),
    StructField("Close", DoubleType()),
    StructField("High", DoubleType()),
    StructField("Low", DoubleType()),
    StructField("Open", DoubleType()),
    StructField("Volume", DoubleType()),
])

df_raw = (spark.read
          .schema(schema_str)
          .option("header", True)
          .csv("/Volumes/workspace/default/capstoneproject/SP500_ETF_Data_V2_Raw.csv"))


In [0]:
df_raw.select("Date").show(10, False)


In [0]:
df_fixed = df_raw.withColumn("Date", F.to_date("Date", "M/d/yyyy"))


In [0]:
df_fixed.createOrReplaceTempView("capstone_typed")


In [0]:
display(df_fixed)

In [0]:
display(
  df_fixed.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("Date").isNull(), 1).otherwise(0)).alias("null_dates"),
    F.sum(F.when(F.col("Close").isNull(), 1).otherwise(0)).alias("null_close"),
    F.sum(F.when(F.col("High").isNull(), 1).otherwise(0)).alias("null_high"),
    F.sum(F.when(F.col("Low").isNull(), 1).otherwise(0)).alias("null_low"),
    F.sum(F.when(F.col("Open").isNull(), 1).otherwise(0)).alias("null_open"),
    F.sum(F.when(F.col("Volume").isNull(), 1).otherwise(0)).alias("null_volume")
  )
)

In [0]:
%sql
SELECT
  Date,
  COUNT(*) AS cnt
FROM capstone_typed
GROUP BY Date
HAVING COUNT(*) > 1;


In [0]:
%sql
SELECT
  Date, Close, High, Low, Open, Volume,
  COUNT(*) AS cnt
FROM capstone_typed
GROUP BY
  Date, Close, High, Low, Open, Volume
HAVING COUNT(*) > 1
ORDER BY cnt DESC;


In [0]:
df_fixed.write.format("delta").mode("overwrite").saveAsTable("SP500_Index_Clean")


In [0]:
%sql
-- Added features:
-- SMA, volatility, OBV, VWAP, CMF
-- plus:
-- relative volume, gap return, 5d/10d/20d momentum,
-- price-to-SMA spreads, SMA crossover spreads, rolling z-score of close

CREATE OR REPLACE TABLE SP500_window_indicators AS
WITH x AS (
  SELECT
    *,
    LAG(Close) OVER (ORDER BY Date) AS prev_close,
    LAG(Volume) OVER (ORDER BY Date) AS prev_volume,

    -- 1-day returns
    (Close / LAG(Close) OVER (ORDER BY Date) - 1.0) AS ret_1d,
    LN(Close / LAG(Close) OVER (ORDER BY Date)) AS log_ret_1d,

    -- Gap return
    (Open / LAG(Close) OVER (ORDER BY Date) - 1.0) AS gap_ret,

    -- Multi-day momentum
    (Close / LAG(Close, 5) OVER (ORDER BY Date) - 1.0) AS mom_5d,
    (Close / LAG(Close, 10) OVER (ORDER BY Date) - 1.0) AS mom_10d,
    (Close / LAG(Close, 20) OVER (ORDER BY Date) - 1.0) AS mom_20d

  FROM SP500_Index_Clean
)

SELECT
  *,

  -- Simple moving averages
  AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)  AS SMA_20,
  AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 49 PRECEDING AND CURRENT ROW)  AS SMA_50,
  AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW) AS SMA_200,

  -- Rolling volatility (20d) and annualized
  STDDEV_SAMP(ret_1d) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS vol_20d,
  STDDEV_SAMP(ret_1d) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) * SQRT(252) AS vol_20d_ann,

  -- Volume SMA
  AVG(Volume) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS Vol_SMA_20,

  -- Relative volume
  Volume / NULLIF(
    AVG(Volume) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW),
    0
  ) AS rel_volume_20,

  -- OBV
  SUM(
    CASE
      WHEN Close > prev_close THEN Volume
      WHEN Close < prev_close THEN -Volume
      ELSE 0
    END
  ) OVER (ORDER BY Date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS OBV,

  -- VWAP (cumulative)
  SUM(((High + Low + Close) / 3.0) * Volume)
    OVER (ORDER BY Date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
  / NULLIF(
      SUM(Volume) OVER (ORDER BY Date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
      0
    ) AS VWAP,

  -- CMF(20)
  (
    SUM(
      (
        ((Close - Low) - (High - Close)) / NULLIF((High - Low), 0)
      ) * Volume
    ) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
  )
  / NULLIF(
      SUM(Volume) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW),
      0
    ) AS CMF_20,

  -- Price-to-SMA spreads
  (Close / NULLIF(AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW), 0) - 1.0)  AS close_vs_sma20,
  (Close / NULLIF(AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 49 PRECEDING AND CURRENT ROW), 0) - 1.0)  AS close_vs_sma50,
  (Close / NULLIF(AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW), 0) - 1.0) AS close_vs_sma200,

  -- SMA crossover spreads
  (
    AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
    /
    NULLIF(AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 49 PRECEDING AND CURRENT ROW), 0)
    - 1.0
  ) AS sma20_vs_sma50,

  (
    AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 49 PRECEDING AND CURRENT ROW)
    /
    NULLIF(AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW), 0)
    - 1.0
  ) AS sma50_vs_sma200,

  (
    AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
    /
    NULLIF(AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW), 0)
    - 1.0
  ) AS sma20_vs_sma200,

  -- Rolling z-score of close (20d)
  (
    Close
    - AVG(Close) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
  )
  / NULLIF(
      STDDEV_SAMP(Close) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW),
      0
    ) AS zclose_20

FROM x
ORDER BY Date;

In [0]:
%sql
SELECT Date, SMA_20, ret_1d, vol_20d, OBV, VWAP, CMF_20
FROM SP500_window_indicators
ORDER BY Date DESC
LIMIT 10;


In [0]:
import pandas as pd
import numpy as np
from pyspark.sql.types import *
from pyspark.sql import functions as F

# Using python to calculate the EMA, MACD, RSI,
# plus ATR_14, Bollinger Bands, and Stochastic Oscillator

base = (
    spark.table("SP500_Index_Clean")
    .select("Date", "Open", "High", "Low", "Close", "Volume")
    .orderBy("Date")
)

schema = StructType([
    StructField("Date", DateType()),

    StructField("EMA_12", DoubleType()),
    StructField("EMA_26", DoubleType()),
    StructField("MACD", DoubleType()),
    StructField("MACD_signal", DoubleType()),
    StructField("MACD_hist", DoubleType()),
    StructField("RSI_14", DoubleType()),

    StructField("ATR_14", DoubleType()),

    StructField("BB_mid_20", DoubleType()),
    StructField("BB_std_20", DoubleType()),
    StructField("BB_upper_20", DoubleType()),
    StructField("BB_lower_20", DoubleType()),
    StructField("BB_width_20", DoubleType()),
    StructField("BB_pctB_20", DoubleType()),

    StructField("stoch_k_14", DoubleType()),
    StructField("stoch_d_3", DoubleType()),
])

def add_recursive(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("Date").copy()

    close = pdf["Close"].astype(float)
    high = pdf["High"].astype(float)
    low = pdf["Low"].astype(float)

    # ----------------------------
    # EMA / MACD
    # ----------------------------
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()

    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    macd_hist = macd - macd_signal

    # ----------------------------
    # RSI(14)
    # ----------------------------
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)

    avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))

    # ----------------------------
    # ATR(14)
    # True Range = max(
    #   High - Low,
    #   abs(High - prev_close),
    #   abs(Low - prev_close)
    # )
    # ----------------------------
    prev_close = close.shift(1)

    tr1 = high - low
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()

    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr_14 = tr.ewm(alpha=1/14, adjust=False).mean()

    # ----------------------------
    # Bollinger Bands (20)
    # ----------------------------
    bb_mid_20 = close.rolling(window=20, min_periods=20).mean()
    bb_std_20 = close.rolling(window=20, min_periods=20).std()

    bb_upper_20 = bb_mid_20 + 2 * bb_std_20
    bb_lower_20 = bb_mid_20 - 2 * bb_std_20

    bb_width_20 = (bb_upper_20 - bb_lower_20) / bb_mid_20.replace(0, np.nan)
    bb_pctB_20 = (close - bb_lower_20) / (bb_upper_20 - bb_lower_20).replace(0, np.nan)

    # ----------------------------
    # Stochastic Oscillator
    # %K(14), %D(3)
    # ----------------------------
    low_14 = low.rolling(window=14, min_periods=14).min()
    high_14 = high.rolling(window=14, min_periods=14).max()

    stoch_k_14 = 100 * (close - low_14) / (high_14 - low_14).replace(0, np.nan)
    stoch_d_3 = stoch_k_14.rolling(window=3, min_periods=3).mean()

    return pd.DataFrame({
        "Date": pdf["Date"],

        "EMA_12": ema12.astype(float),
        "EMA_26": ema26.astype(float),
        "MACD": macd.astype(float),
        "MACD_signal": macd_signal.astype(float),
        "MACD_hist": macd_hist.astype(float),
        "RSI_14": rsi.astype(float),

        "ATR_14": atr_14.astype(float),

        "BB_mid_20": bb_mid_20.astype(float),
        "BB_std_20": bb_std_20.astype(float),
        "BB_upper_20": bb_upper_20.astype(float),
        "BB_lower_20": bb_lower_20.astype(float),
        "BB_width_20": bb_width_20.astype(float),
        "BB_pctB_20": bb_pctB_20.astype(float),

        "stoch_k_14": stoch_k_14.astype(float),
        "stoch_d_3": stoch_d_3.astype(float),
    })

def iterator_fn(iterator):
    pdf = pd.concat(list(iterator), ignore_index=True)
    yield add_recursive(pdf)

recursive = base.mapInPandas(iterator_fn, schema=schema)

display(recursive.orderBy(F.desc("Date")))

In [0]:
windowed = spark.table("SP500_window_indicators")
final = windowed.join(recursive, on="Date", how="left")

(final.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("SP500_with_indicators"))


In [0]:
data_path = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/SP500_with_indicators"

(final.write
 .format("delta")
 .mode("overwrite")
 .save(data_path))


In [0]:
from pyspark.sql import functions as F

csv_dir = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/exports/SP500_with_indicators_csv"

# Export as single CSV file
(final
 .coalesce(1)              
 .write
 .mode("overwrite")        
 .option("header", True)   # include column names
 .csv(csv_dir)
)


In [0]:
files = dbutils.fs.ls(csv_dir)
csv_part = [f.path for f in files if f.name.endswith(".csv")][0]

final_csv_path = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/exports/S&P500_with_indicators_V2.csv"

dbutils.fs.mv(csv_part, final_csv_path)
dbutils.fs.rm(csv_dir, recurse=True)

print("CSV saved to:", final_csv_path)


In [0]:
import mlflow

print("Spark Version:", spark.version)
print("MLflow Version:", mlflow.__version__)